In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)


In [ ]:
import random
random.seed(42)
kisiler=["zıpırcan","tripcan","ilgisu","nazlıhan","berkay","sarı saruhan","saruhan","heymen","aren","setenay"]
zamanlar=["sabah","öğle","akşam","dün","bugün"]
yer_e=["parka","okula","eve","kütüphaneye","denize","pazara","bahçede"]
yer_de=["parkta","okulda","evde","kütüphanede","denizde","pazarda","bahçede"]
isler=[("kitap","okudu"),("çay","içti"),("top","oynadı"),("resim","yaptı")]
havalar=["güzel","soğuk","sıcak","yağmurlu","günesli","rüzgarlı"]

sifatlar=["mutlu","yorgun","neşeli","sessiz","heyecanlı"]

def cumle():
  k,z=random.choice(kisiler),random.choice(zamanlar)
  i=random.randrange(len(yer_e))
  nesne,fiil=random.choice(isler)
  kalip=random.randrange(5)
  if kalip == 0: return f"{z} {k} {yer_e[i]} gitti"
  if kalip == 1: return f"{z} {k} {yer_de[i]} {nesne} {fiil}"
  if kalip == 2: return f"{z} hava çok {random.choice(havalar)}"
  if kalip == 3: return f"bugün çok {random.choice(sifatlar)}"
  return f"{z} {k} {yer_de[i]} {nesne} {fiil}"

metin= "\n".join(cumle() for _ in range(3000))
print("metin uzunluğu",len(metin))
print(metin[:245])


metin uzunluğu 77844
sabah tripcan pazarda top oynadı
öğle nazlıhan pazarda kitap okudu
bugün tripcan kütüphaneye gitti
öğle tripcan okulda kitap okudu
bugün çok heyecanlı
bugün çok neşeli
öğle berkay okula gitti
dün hava çok günesli
sabah berkay pazarda resim yaptı


In [ ]:
harfler=sorted(set(metin))
SOZLUK=len(harfler)
harf2no={u:i for i,u in enumerate(harfler)}
no2harf={i:u for i,u in enumerate(harfler)}
def kodla(s):
  return [harf2no[h]for h in s]

def coz(sayilar):
  return "".join(no2harf[i] for i in sayilar)

print("sozluk boyutu",SOZLUK)
print("harfler:","".join(harfler))

sozluk boyutu 29
harfler: 
 abcdeghiklmnoprstuvyzçöüğış


In [ ]:
veri=torch.tensor(kodla(metin))
n=int(0.9*len(veri)/10)
egitim_veri=veri[:n]
dogrulama_veri=veri[n:]


In [ ]:
baglam=64
yıgın=32

def yigin_al(bolum):
  d=egitim_veri if bolum =="egitim" else dogrulama_veri
  bas = torch.randint(len(d)-baglam-1,(yıgın,))
  x=torch.stack([d[i:i+baglam] for i in bas])
  y=torch.stack([d[i+1:i+baglam+1] for i in bas])
  return x,y
x,y=yigin_al("egitim")

In [ ]:
class MaskeliDikkat(nn.Module):
  def __init__(self,d_model,n_kafa):
    super().__init__()
    self.n_kafa=n_kafa
    self.d_k=d_model//n_kafa
    self.qkv=nn.Linear(d_model,3*d_model,bias=False)
    self.cikis=nn.Linear(d_model,d_model,bias=False)
    self.register_buffer("maske",torch.tril(torch.ones(baglam,baglam)))
  def forward(self,x):
    B,L,D=x.shape
    q,k,v=self.qkv(x).split(D,dim=2)
    q=q.view(B,L,self.n_kafa,self.d_k).transpose(1,2)
    k=k.view(B,L,self.n_kafa,self.d_k).transpose(1,2)
    v=v.view(B,L,self.n_kafa,self.d_k).transpose(1,2)
    skor=q@k.transpose(-2,-1)/math.sqrt(self.d_k)
    # Maskeyi dizilim uzunluğu olan L'ye göre dilimliyoruz
    skor=skor.masked_fill(self.maske[None,None,:L,:L]==0,float("-inf"))
    agirlik=F.softmax(skor,dim=-1)
    cikti=(agirlik@v).transpose(1,2).reshape(B,L,D)
    return self.cikis(cikti)

In [ ]:
from torch.autograd import forward_ad
class GPTBlok(nn.Module):
  def __init__(self,d_model,n_kafa):
    super().__init__()
    self.norm1=nn.LayerNorm(d_model)
    self.dikkat=MaskeliDikkat(d_model,n_kafa)
    self.norm2=nn.LayerNorm(d_model)

    self.ffn=nn.Sequential(
      nn.Linear(d_model,4*d_model),
      nn.ReLU(),
      nn.Linear(4*d_model,d_model),
      nn.Dropout(0.5)
    )

  def forward(self,x):
    x=x+self.dikkat(self.norm1(x))
    x=x+self.ffn(self.norm2(x))
    return x

class MiniGPT(nn.Module):
  def __init__(self,d_model=64,n_kafa=4,n_katman=2):
    super().__init__()
    self.harf_gomme = nn.Embedding(SOZLUK,d_model)
    self.konum_gomme=nn.Embedding(baglam,d_model)
    self.blok=nn.Sequential(*[GPTBlok(d_model,n_kafa) for _ in range(n_katman)])
    self.norm=nn.LayerNorm(d_model)
    self.cikis=nn.Linear(d_model,SOZLUK)

  def forward(self,x,hedef=None):
    B,L=x.shape
    h=self.harf_gomme(x)+self.konum_gomme(torch.arange(L))
    h=self.norm(self.blok(h))
    logits=self.cikis(h)
    if hedef is None:
      return logits
    loss=F.cross_entropy(logits.view(-1,SOZLUK),hedef.view(-1))
    return logits,loss

  @torch.no_grad()
  def uret(self,baslangic,n_harf,sicaklik=0.8):
    x=torch.tensor(kodla(baslangic)).unsqueeze(0)
    for _ in range(n_harf):
      x_kirp=x[:,-baglam:]
      logits=self(x_kirp)
      logits=logits[:,-1,:]/sicaklik
      olasilik=F.softmax(logits,dim=-1)
      sonraki=torch.multinomial(olasilik,num_samples=1)
      x=torch.cat((x,sonraki),dim=1)
    return coz(x[0].tolist())

In [ ]:
model=MiniGPT()

In [ ]:
optimizer=torch.optim.AdamW(model.parameters(),lr=3e-3)
egitim_kayip,dogrulama_kayip,adimlar=[],[],[]

for adim in range(1,1501):
  model.train()
  x,y=yigin_al("egitim")

  _,kayip=model(x,y)
  optimizer.zero_grad()
  kayip.backward()
  optimizer.step()

  if adim %100==0:
    model.eval()
    with torch.no_grad():
      _,d_kayip=model(*yigin_al("dogrulama"))
      adimlar.append(adim)
      egitim_kayip.append(kayip.item())
      dogrulama_kayip.append(d_kayip.item())
      if adim %300 == 0:
        print(f"Adım {adim:4d} | egitim kaybı {kayip.item():.3f} | dogrulama kaybı {d_kayip.item():.3f}")

Adım  300 | egitim kaybı 0.334 | dogrulama kaybı 0.348
Adım  600 | egitim kaybı 0.274 | dogrulama kaybı 0.332
Adım  900 | egitim kaybı 0.246 | dogrulama kaybı 0.384
Adım 1200 | egitim kaybı 0.213 | dogrulama kaybı 0.412
Adım 1500 | egitim kaybı 0.197 | dogrulama kaybı 0.480


In [ ]:
model.eval()
for basla in ["bugün","ilgisu","yorgun"]:
  print(model.uret(basla,150))
  print("--"*50)

bugün çok heyecanlı
akşam aren bahçede gitti
bugün berkay okulda resim yaptı
sabah setenay parkta çay içti
sabah zıpırcan bahçede top oynadı
bugün ilgisu d
----------------------------------------------------------------------------------------------------
ilgisu parkta kitap okudu
akşam hava çok soğuk
bugün heymen kütüphaneye gitti
akşam hava çok rüzgarlı
ün bugün çok mutlu
öğle hava çok sıcak
bugün berkay kü
----------------------------------------------------------------------------------------------------
yorgun
bugün çok neşeli
akşam ilgisu evde top oynadı
sabah berkay parkta çay içti
sabah setenay bahçede top oynadı
bugün ilgisu okulda kitap okudu
bugün çok
----------------------------------------------------------------------------------------------------
